In [1]:
# %% ../nbs/00_core.ipynb 3
from IPython.display import display, HTML
from fasthtml.common import *
from fasthtml.jupyter import *
from uuid import uuid4
from middleware import *

# %% ../nbs/00_core.ipynb 5
def HShow(comp, iframe_height="auto", app=None, port=8000):
    route = f'/{uuid4()}'
    app.get(route)(lambda: comp)
    display(HTML(f'<a href="http://localhost:{port}{route}" target="_blank">Open in new tab</a>'))
    return HTMX(route,port=port,iframe_height=iframe_height)

# %% ../nbs/00_core.ipynb 6
def create_server(app, stop_server=True,server_varname='server'):
    if stop_server and server_varname in globals(): globals()[server_varname].stop()
    for port in range(8000,8030):                   
        if is_port_free(port):
            server = JupyUvi(app, port=port)
            Show = partial(HShow, app=app, port=port)
            return server, Show

In [2]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union, Optional

@dataclass
class JSXNode:
    """Represents a JSX element"""
    tag: str
    props: Dict[str, Any]
    children: List[Any]
    
    def to_jsx(self) -> str:
        """Convert to JSX string"""
        props_str = " ".join(
            f'{k}={json.dumps(v)}' if isinstance(v, (str, int, float, bool)) 
            else f'{k}={{{json.dumps(v)}}}' 
            for k, v in self.props.items()
        )
        
        children_str = "".join(
            child.to_jsx() if isinstance(child, JSXNode)
            else json.dumps(child) if isinstance(child, (str, int, float, bool))
            else str(child)
            for child in self.children
        )
        
        return f"<{self.tag} {props_str}>{children_str}</{self.tag}>"

class ReactIsland:
    """A React island that can render FT components as React"""
    def __init__(self, 
                 children: Any,
                 component: str = "Island",
                 hydrate: bool = True):
        self.children = children
        self.component = component
        self.hydrate = hydrate
        
    def ft_to_jsx(self, ft) -> JSXNode:
        """Convert FT element to JSX"""
        if isinstance(ft, (str, int, float, bool)):
            return ft
            
        tag, children, attrs = ft
        
        # Transform FT attributes to React props
        props = {}
        for k, v in attrs.items():
            # Handle special cases
            if k == "cls":
                props["className"] = v
            elif k.startswith("on_"):
                # Convert Python event handlers to React ones
                event = k[3:].capitalize()
                props[f"on{event}"] = v
            else:
                props[k.replace("_", "-")] = v
                
        # Transform children recursively
        jsx_children = [
            self.ft_to_jsx(child) if isinstance(child, (list, tuple))
            else child
            for child in children
        ]
        
        # Map FT tags to React components
        component_map = {
            "Button": "ShadcnButton",
            "Input": "ShadcnInput",
            # Add more mappings as needed
        }
        
        react_tag = component_map.get(tag, tag.lower())
        return JSXNode(react_tag, props, jsx_children)

    def __ft__(self):
        jsx = self.ft_to_jsx(self.children)
        
        # Create a container with React hydration data
        return Div(
            NotStr(jsx.to_jsx()),
            id=f"island-{id(self)}",
            data_component=self.component,
            data_hydrate=json.dumps(self.hydrate),
            cls="react-island"
        )

# Example usage
@rt('/')
def get():
    return Main(
        # Regular FT syntax, but will be rendered as React
        ReactIsland(
            Div(
                H1("Hello from React", cls="title"),
                Button(
                    "Click me",
                    variant="outline",
                    on_click="() => alert('Clicked!')"
                ),
                Input(
                    placeholder="Enter text",
                    on_change="(e) => console.log(e.target.value)"
                )
            )
        )
    )